# SaaS Sales — Exploratory Analysis

**Dataset:** Amazon AWS SaaS Sales — https://www.kaggle.com/datasets/nnthanh101/aws-saas-sales

This notebook explores the data interactively. The production logic lives in `src/pipeline.py`;
this is for investigation, not the source of truth. Run the pipeline first so the cleaned
dataset exists.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from pipeline import (
    breakdown, build_order_level, clean_column_names, clean_data,
    engineer_features, headline_kpis, load_data, top_customers,
)

pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load and clean

Reuse the pipeline's functions so the notebook and production code cannot drift apart.

In [ ]:
raw = load_data()
df = engineer_features(clean_data(clean_column_names(raw)))
df = df.drop(columns=[c for c in ["row_id", "license"] if c in df.columns])
order_df = build_order_level(df)
df.head()

## 2. Headline KPIs

Note `n_orders` uses a **distinct** count — one order spans multiple rows.

In [ ]:
kpis = headline_kpis(df, order_df)
pd.Series(kpis).to_frame("value")

## 3. Where does profit come from?

In [ ]:
breakdown(df, "region")

In [ ]:
breakdown(df, "product").sort_values("margin_pct")

## 4. The discount story

The core finding: margin erodes as discount rises.

In [ ]:
bucket = pd.cut(df["discount"], bins=[-0.01, 0, 0.10, 0.20, 0.30, 1.0],
                labels=["0%", "1-10%", "11-20%", "21-30%", "30%+"])
margin_by_bucket = (df.groupby(bucket, observed=True)
                      .apply(lambda g: g["profit"].sum() / g["sales"].sum() * 100,
                             include_groups=False)
                      .rename("margin_pct"))
margin_by_bucket.to_frame()

In [ ]:
ax = margin_by_bucket.plot(kind="bar", figsize=(7, 4),
                          color=["#2a9d6f" if v > 0 else "#c0504d" for v in margin_by_bucket])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Profit Margin % by Discount Bucket")
ax.set_ylabel("Margin %")
plt.show()

## 5. Revenue concentration

How dependent is the business on its largest customers?

In [ ]:
tc = top_customers(df, 10)
print(f"Top 10 customers = {tc['pct_of_total'].sum():.1f}% of revenue")
tc

## 6. Next steps

Findings from this notebook feed `reports/business_report.md`. The dashboard build is
documented in `dashboards/` — see the PDF guide.

**Reminder:** if you ran this on the synthetic sample, the numbers are illustrative.
Use the real Kaggle file for anything you present.